# C1.1 — Unsupervised matching, repaired: the entropic regularizer, and
# why whitening makes it worse

C1 recovered 8.6 per cent exact matches from geometry alone against a
chance rate of 0.067 per cent — 129x chance, real signal — but the
resulting map reached only 6 per cent of ceiling in retrieval, against
88 per cent for the supervised map. Two things were wrong with the run,
and one plausible fix turns out to be actively harmful. Both were
measured on the project's own `pairs.npz` before this notebook was
written.

**1. The entropic regularizer was too small.** C1 used `epsilon=5e-4`
with `max_iter=200`; Sinkhorn did not converge, and an unconverged
coupling is a poorly-optimised one. Measured on real MobileCLIP/SigLIP
vectors at N = 800:

| setting | exact matching | time |
|---|---|---|
| `epsilon=5e-4, max_iter=200` (C1) | 10.2% | 105 s |
| `epsilon=5e-3, max_iter=1000` | **33.5%** | **7 s** |

Three times the accuracy and fifteen times faster. Too small an epsilon
makes each Sinkhorn projection nearly a hard assignment, which both
converges slowly and locks in early mistakes.

**2. ICP degraded the solution monotonically** — 8.6, 8.3, 7.9, ..., 6.9
per cent, converging stably at a worse answer. Fitting a map from mostly
wrong assignments teaches the wrong correspondence, which is then
re-matched onto more confidently. Convergence there means agreement with
its own errors. This notebook reports GW alone and treats ICP as a
documented negative.

**3. Whitening first — the obvious idea from Section C.11 — makes it far
worse.** Section C.11 found that whitening the target space rescues
cosine retrieval. Applying the same transform before GW collapses
matching to near chance (measured, N = 400):

| | exact matching |
|---|---|
| raw spaces | **50.5%** |
| whitened spaces | 0.5% (chance = 0.25%) |

**This is not a contradiction — it is the sharpest distinction in the
project.** Cosine retrieval reads a correspondence that is already
known, and there uneven variance is noise that drowns the signal.
Gromov-Wasserstein *discovers* an unknown correspondence by matching one
space's distance structure to the other's — and there uneven variance
**is** the signal: it is what makes a point's position distinctive
relative to its neighbours. Whitening equalises every direction and so
erases exactly the structure GW needs. Isotropy helps you *read* a
correspondence and hurts you when you must *find* one.

In [ ]:
# ---- environment (identical to the other notebooks) ----
import os
from pathlib import Path
STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR is unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - using LOCAL_DIR instead")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"]); DATA_DIR.mkdir(parents=True,
                                                        exist_ok=True)
print("DATA_DIR:", DATA_DIR)

In [ ]:
!pip -q install pot

import importlib
assert importlib.util.find_spec("ot"), "POT did not install - re-run"
import ot
print("POT", ot.__version__)

In [ ]:
import numpy as np, time

N       = 1500          # matched set size; chance = 1/N
EPS_GRID = [1e-3, 3e-3, 5e-3, 1e-2]   # the knob C1 got wrong
MAX_ITER = 1500
rng = np.random.default_rng(0)

def l2n(V):
    return V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-12)

pairs = np.load(str(DATA_DIR / "pairs.npz"))
idx = rng.choice(len(pairs["mob_img"]), N, replace=False)
A = l2n(pairs["mob_img"][idx].astype(np.float64))    # phone space
B = l2n(pairs["sig_img"][idx].astype(np.float64))    # server space

# the pairing is destroyed: row i of A and row i of B are the same image,
# but nothing below uses that fact until the final scoring line.
print(f"{N} vectors per side; chance matching = {100/N:.3f}%")

def effrank(X):
    Xc = X - X.mean(0)
    s = np.linalg.svd(Xc, full_matrices=False, compute_uv=False)
    p = s ** 2 / (s ** 2).sum(); p = p[p > 0]
    return float(np.exp(-(p * np.log(p)).sum()))
print(f"effective rank  phone {effrank(A):.0f}/{A.shape[1]}   "
      f"server {effrank(B):.0f}/{B.shape[1]}")

## Stage 1 — Gromov-Wasserstein, swept over the entropic regularizer

In [ ]:
def gw(A, B, eps, max_iter=MAX_ITER):
    CA = 1.0 - A @ A.T
    CB = 1.0 - B @ B.T
    CA /= CA.mean(); CB /= CB.mean()      # scale-normalise the two shapes
    p = np.full(len(A), 1/len(A))
    return ot.gromov.entropic_gromov_wasserstein(
        CA, CB, p, p, "square_loss", epsilon=eps, max_iter=max_iter,
        tol=1e-9, verbose=False)

def score(G, k=(1, 5, 10)):
    n = len(G); truth = np.arange(n)
    order = np.argsort(-G, axis=1)
    rank = (order == truth[:, None]).argmax(1)
    return {kk: float((rank < kk).mean()) for kk in k}

print(f"{'epsilon':>9s} {'exact':>8s} {'top-5':>8s} {'top-10':>8s} "
      f"{'x chance':>9s} {'secs':>6s}")
best = (None, -1, None)
for eps in EPS_GRID:
    t0 = time.time(); G = gw(A, B, eps); s = score(G)
    print(f"{eps:9.0e} {s[1]*100:7.1f}% {s[5]*100:7.1f}% {s[10]*100:7.1f}% "
          f"{s[1]*N:8.0f}x {time.time()-t0:6.0f}")
    if s[1] > best[1]:
        best = (eps, s[1], G)
EPS_BEST, G_BEST = best[0], best[2]
print(f"\nbest epsilon {EPS_BEST:.0e} at {best[1]*100:.1f}% exact "
      f"({best[1]*N:.0f}x chance)")

## The whitening control — a deliberate negative

Run once to confirm on your own data that isotropy hurts *discovery*
even though Section C.11 showed it rescues *reading*. This is the
cell that makes the distinction empirical rather than asserted.

In [ ]:
def whiten(X):
    Xc = X - X.mean(0)
    C = np.cov(Xc.T) + 1e-6 * np.eye(X.shape[1])
    ev, V = np.linalg.eigh(C); ev = np.clip(ev, 1e-8, None)
    return Xc @ (V @ np.diag(ev ** -0.5) @ V.T)

Aw, Bw = l2n(whiten(A)), l2n(whiten(B))
print(f"effective rank after whitening: phone {effrank(Aw):.0f}, "
      f"server {effrank(Bw):.0f}  (was {effrank(A):.0f}, {effrank(B):.0f})")
s_w = score(gw(Aw, Bw, EPS_BEST))
print(f"\n{'spaces':12s} {'exact matching':>15s}")
print(f"{'raw':12s} {best[1]*100:14.1f}%")
print(f"{'whitened':12s} {s_w[1]*100:14.1f}%   (chance {100/N:.3f}%)")
print("\nGW matches STRUCTURE to STRUCTURE; uneven variance is what makes")
print("a point distinctive relative to its neighbours, so equalising it")
print("removes the signal. Cosine retrieval reads a KNOWN correspondence,")
print("where the same unevenness is noise. Opposite prescriptions, one")
print("underlying geometry - and both were measured, not assumed.")

## Stage 2 — the map, and how good a DECIDER it is

Two questions, deliberately separated (the distinction of Appendix D.2):
retrieval asks whether the map is usable for search; verification AUC
asks whether a threshold on its output can tell a true pair from a
random one. A map can be far too weak for the first and still carry
real signal on the second.

In [ ]:
# fit a ridge map from the UNSUPERVISED assignment only
assign = G_BEST.argmax(1)
Xs, Ys = A, B[assign]                      # pairing proposed by GW
W_uns = np.linalg.solve(Xs.T @ Xs + 1e-2 * np.eye(Xs.shape[1]), Xs.T @ Ys)

ad = np.load(str(DATA_DIR / "adapter.npz"))
te = ad["eval_idx"]
mob_e = pairs["mob_img"][te].astype(np.float64)
sig_e = l2n(pairs["sig_img"][te].astype(np.float64))
txt_e = l2n(pairs["sig_txt"][te].astype(np.float64))

def recall(S):
    o = np.argsort(-S, 1); r = (o == np.arange(len(S))[:, None]).argmax(1)
    return {k: float((r < k).mean()) for k in (1, 5, 10)}

def auc(pos, neg):
    y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]; s = np.r_[pos, neg]
    o = np.argsort(-s); y = y[o]
    tpr = np.cumsum(y)/y.sum(); fpr = np.cumsum(1-y)/(1-y).sum()
    return float(np.trapezoid(tpr, fpr))

rng2 = np.random.default_rng(1)
j = rng2.permutation(len(te)); j = np.where(j == np.arange(len(te)),
                                            (j+1) % len(te), j)
print(f"{'map':34s} {'R@1':>6s} {'R@5':>6s} {'R@10':>6s} {'verif AUC':>10s}")
for lab, M in [("ceiling (SigLIP native)", sig_e),
               ("supervised W (3,000 true pairs)",
                l2n(mob_e @ ad["W_ridge"].astype(np.float64))),
               ("UNSUPERVISED W (0 pairs)", l2n(mob_e @ W_uns))]:
    r = recall(txt_e @ M.T)
    a = auc((M * sig_e).sum(1), (M * sig_e[j]).sum(1)) if lab[0] != "c" \
        else 1.0
    print(f"{lab:34s} {r[1]:6.3f} {r[5]:6.3f} {r[10]:6.3f} {a:10.3f}")
print("\nverification AUC asks a WEAKER question than retrieval: it needs")
print("only to separate a true pair from a typical random one, not to")
print("outrank 999 impostors. A map can fail the second and pass the")
print("first, and reporting both is the honest description.")

## How to read this

The headline number is exact matching against chance, from Stage 1: the
correspondence between two independently trained encoders is partially
recoverable **with no paired examples at all**, from the shapes of the
two point clouds. Whatever that figure reaches, it is the most demanding
form of the project's thesis, and the supervised map remains far better
— unsupervised recovery is evidence about the geometry, not a
deployable component.

Three results here are negative and are reported as such: ICP degrades
the GW solution, whitening destroys it, and the resulting map is far
below the supervised one in retrieval. Negative results with a measured
mechanism are worth more than a tuned number without one.